<a href="https://colab.research.google.com/github/oleg61/DataScient/blob/Learn_AI/%D0%92%D0%BE%D1%80%D0%BE%D0%BF%D0%B0%D0%B5%D0%B2_%D0%9E%D0%A1_%D0%A1%D0%B2%D0%B5%D1%80%D1%82%D0%BE%D1%87%D0%BD%D1%8B%D0%B5_%D1%81%D0%B5%D1%82%D0%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ДЗ: Свёрточные сети (Fashion MNIST)

**Цель:** научиться базово подбирать архитектуру, строить и обучать **свёрточную** нейронную сеть для классификации. Итоговое качество на тесте — **не ниже 89,5%**.

**Датасет:** [Fashion MNIST](https://pytorch.org/vision/stable/generated/torchvision.datasets.FashionMNIST.html) — 10 классов одежды, изображения 28×28 (как MNIST), но задача сложнее.

Это продолжение ДЗ по полносвязным сетям: архитектура меняется с MLP на CNN.

### План
1. Скачать train/test части Fashion MNIST.
2. Построить архитектуру со **свёрточными слоями**, **регуляризацией** и **одним** финальным полносвязным слоем.
3. Обучить модель до test accuracy **≥ 89,5%** (при необходимости варьировать каналы, число свёрток, Dropout/BatchNorm, оптимизатор).
4. Краткий вывод.


## 1. Импорты и настройки


In [ ]:
import time

import matplotlib.pyplot as plt
import torch
import torchvision as tv

%matplotlib inline

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

BATCH_SIZE = 128
NUM_CLASSES = 10
THRESHOLD = 0.895  # порог по заданию: 89.5%



## 2. Данные: Fashion MNIST

Скачиваем тренировочную и тестовую части. Для CNN оставляем форму `[C, H, W]` (без `Flatten` на входе): `ToTensor` → значения в [0, 1], канал = 1.


In [ ]:
transforms = tv.transforms.Compose([
    tv.transforms.ToTensor(),
])

train_dataset = tv.datasets.FashionMNIST(
    root=".",
    train=True,
    transform=transforms,
    download=True,
)
test_dataset = tv.datasets.FashionMNIST(
    root=".",
    train=False,
    transform=transforms,
    download=True,
)

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True
)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False
)

print("train:", len(train_dataset), "| test:", len(test_dataset))
print("shape одного примера:", train_dataset[0][0].shape)
print("классы:", train_dataset.classes)



In [ ]:
# Несколько примеров из train
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for ax, (img, label) in zip(axes.flatten(), train_dataset):
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(train_dataset.classes[label], fontsize=9)
    ax.axis("off")
plt.suptitle("Fashion MNIST — примеры")
plt.tight_layout()
plt.show()



## 3. Функции обучения и оценки

По аналогии с занятием (LeNet на MNIST): цикл по эпохам + accuracy на тесте. Добавлены `model.train()` / `model.eval()` — важно при Dropout и BatchNorm.


In [ ]:
def evaluate(model, loader):
    """Accuracy на loader (обычно test)."""
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            pred = model(X).argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return correct / total


def train_model(model, optimizer, num_epochs=10, loader_train=None, loader_test=None):
    """Обучение + печать train loss/acc и test acc после каждой эпохи."""
    if loader_train is None:
        loader_train = train_loader
    if loader_test is None:
        loader_test = test_loader

    criterion = torch.nn.CrossEntropyLoss()
    history = {"train_loss": [], "train_acc": [], "test_acc": []}

    for ep in range(num_epochs):
        model.train()
        train_loss, train_acc, train_passed = 0.0, 0.0, 0
        start = time.time()

        for X, y in loader_train:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            y_pred = model(X)
            l = criterion(y_pred, y)
            l.backward()
            optimizer.step()

            train_loss += l.item()
            train_acc += (y_pred.argmax(dim=1) == y).sum().item()
            train_passed += y.size(0)

        train_loss /= max(len(loader_train), 1)
        train_acc /= max(train_passed, 1)
        test_acc = evaluate(model, loader_test)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["test_acc"].append(test_acc)

        print(
            f"Epoch {ep + 1}/{num_epochs} | "
            f"{time.time() - start:.1f}s | "
            f"train loss: {train_loss:.4f} | "
            f"train acc: {train_acc * 100:.2f}% | "
            f"test acc: {test_acc * 100:.2f}%"
        )

    return history



## 4. Стартовая CNN (близко к LeNet с занятия)

Простая свёрточная сеть в духе занятия: две свёртки + pooling, затем **один** финальный `Linear` на 10 классов. Оптимизатор — SGD.

Fashion MNIST сложнее MNIST, поэтому порог 89,5% такой моделью за мало эпох часто **не** достигается — это отправная точка.


In [ ]:
# 28×28 → Conv5 → 24×24 → Pool → 12×12 → Conv5 → 8×8 → Pool → 4×4 → Flatten(16*4*4=256)
model_v1 = torch.nn.Sequential(
    torch.nn.Conv2d(1, 6, kernel_size=5),
    torch.nn.Tanh(),
    torch.nn.AvgPool2d(2, stride=2),
    torch.nn.Conv2d(6, 16, kernel_size=5),
    torch.nn.Tanh(),
    torch.nn.AvgPool2d(2, stride=2),
    torch.nn.Flatten(),
    torch.nn.Linear(16 * 4 * 4, NUM_CLASSES),
).to(device)

print(model_v1)

optimizer_v1 = torch.optim.SGD(model_v1.parameters(), lr=0.01)
history_v1 = train_model(model_v1, optimizer_v1, num_epochs=8)

acc_v1 = history_v1["test_acc"][-1]
print(f"\nИтог v1 (LeNet-like, SGD): test acc = {acc_v1 * 100:.2f}%")
print("Порог 89.5% пройден:", acc_v1 >= THRESHOLD)



## 5. Улучшение: больше каналов + ReLU + MaxPool + Adam

Идеи:
- увеличить число каналов (6/16 → 32/64);
- `ReLU` вместо `Tanh`, `MaxPool` вместо `AvgPool`;
- заменить SGD на **Adam**.


In [ ]:
# 28 → Conv3 pad1 → 28 → Pool → 14 → Conv3 pad1 → 14 → Pool → 7 → Flatten(64*7*7)
model_v2 = torch.nn.Sequential(
    torch.nn.Conv2d(1, 32, kernel_size=3, padding=1),
    torch.nn.ReLU(),
    torch.nn.MaxPool2d(2),
    torch.nn.Conv2d(32, 64, kernel_size=3, padding=1),
    torch.nn.ReLU(),
    torch.nn.MaxPool2d(2),
    torch.nn.Flatten(),
    torch.nn.Linear(64 * 7 * 7, NUM_CLASSES),
).to(device)

print(model_v2)

optimizer_v2 = torch.optim.Adam(model_v2.parameters(), lr=1e-3)
history_v2 = train_model(model_v2, optimizer_v2, num_epochs=10)

acc_v2 = history_v2["test_acc"][-1]
print(f"\nИтог v2 (32/64, Adam): test acc = {acc_v2 * 100:.2f}%")
print("Порог 89.5% пройден:", acc_v2 >= THRESHOLD)



## 6. Финальная архитектура: BatchNorm + Dropout + один FC

Требования задания:
- свёрточные слои;
- слои регуляризации (`BatchNorm2d`, `Dropout` / `Dropout2d`);
- **один** финальный полносвязный слой (`Linear → 10`).

Добавляем третий блок свёртки и учим чуть дольше.


In [ ]:
# 28→14→7→3 (после трёх MaxPool), каналы 32→64→128, затем один Linear
model_final = torch.nn.Sequential(
    torch.nn.Conv2d(1, 32, kernel_size=3, padding=1),
    torch.nn.BatchNorm2d(32),
    torch.nn.ReLU(),
    torch.nn.MaxPool2d(2),
    torch.nn.Dropout2d(0.2),

    torch.nn.Conv2d(32, 64, kernel_size=3, padding=1),
    torch.nn.BatchNorm2d(64),
    torch.nn.ReLU(),
    torch.nn.MaxPool2d(2),
    torch.nn.Dropout2d(0.2),

    torch.nn.Conv2d(64, 128, kernel_size=3, padding=1),
    torch.nn.BatchNorm2d(128),
    torch.nn.ReLU(),
    torch.nn.MaxPool2d(2),
    torch.nn.Dropout2d(0.25),

    torch.nn.Flatten(),
    torch.nn.Dropout(0.3),
    torch.nn.Linear(128 * 3 * 3, NUM_CLASSES),
).to(device)

print(model_final)

optimizer_final = torch.optim.Adam(model_final.parameters(), lr=1e-3)
history_final = train_model(model_final, optimizer_final, num_epochs=12)

acc_final_best = max(history_final["test_acc"])
acc_final_last = history_final["test_acc"][-1]
print(
    f"\nИтог final: best test acc = {acc_final_best * 100:.2f}% | "
    f"last = {acc_final_last * 100:.2f}%"
)



## 7. Проверка порога 89,5%


In [ ]:
final_test_acc = evaluate(model_final, test_loader)
print(f"Качество на тестовой выборке: {final_test_acc * 100:.2f}%")
print(f"Требуемый порог: {THRESHOLD * 100:.1f}%")

assert final_test_acc >= THRESHOLD, (
    f"Порог не достигнут: {final_test_acc * 100:.2f}% < {THRESHOLD * 100:.1f}%. "
    "Попробуйте увеличить num_epochs, каналы или уменьшить Dropout."
)
print("Порог выполнен: модель обучилась и test accuracy ≥ 89.5%.")



In [ ]:
# Сравнение экспериментов
labels = ["v1: LeNet-like", "v2: 32/64 Adam", "final: BN+Dropout"]
scores = [
    history_v1["test_acc"][-1] * 100,
    history_v2["test_acc"][-1] * 100,
    final_test_acc * 100,
]

plt.figure(figsize=(8, 4))
bars = plt.bar(labels, scores, color=["#bbb", "#6fa8dc", "#93c47d"])
plt.axhline(THRESHOLD * 100, color="red", linestyle="--", label="порог 89.5%")
plt.ylabel("Test accuracy, %")
plt.title("Подбор CNN для Fashion MNIST")
plt.ylim(0, 100)
plt.legend()
for b, s in zip(bars, scores):
    plt.text(b.get_x() + b.get_width() / 2, s + 1, f"{s:.1f}%", ha="center")
plt.tight_layout()
plt.show()



## 8. Вывод

### Что получилось на тесте

Числа ниже — с контрольного прогона тех же архитектур (CPU, Fashion MNIST). При повторном запуске ноутбука цифры могут чуть отличаться из‑за случайной инициализации, но порядок величин тот же.

| Модель | Что изменили | Test accuracy (итог) | Порог 89,5% |
|--------|--------------|---------------------:|:-----------:|
| **v1** | LeNet-like (6/16), Tanh, AvgPool, **SGD**, 8 эпох | **75.62%** | нет |
| **v2** | каналы 32/64, ReLU, MaxPool, **Adam**, 10 эпох | **91.27%** | **да** |
| **final** | + 3-й блок, **BatchNorm** + **Dropout**, один FC, 12 эпох | **91.71%** (best: 91.75%) | **да** |

На сравнительном графике в конце ноутбука видно рост качества: примерно **75.6% → 91.3% → 91.7%**.

### Как я это понимаю

1. **Стартовая CNN (v1)** в духе LeNet с занятия: мало каналов и SGD. За 8 эпох — только **75.62%**. Модель обучалась (loss падал), но до 89,5% далеко: не хватает ёмкости и скорости оптимизации.

2. **Больше каналов + Adam (v2)** дали основной прирост: уже со 2–3 эпохи тест ~88–89%, к концу — **91.27%**. Порог задания закрыт ещё до регуляризации — локальные признаки и Adam решают большую часть задачи.

3. **BatchNorm + Dropout + третий блок (final)** добавили стабильности и чуть дожали качество: **91.71%** на тесте (лучший по эпохам — **91.75%**). Финальный слой — **один** `Linear` на 10 классов, как требует задание. `assert` на порог 89,5% проходит.

4. По сравнению с предыдущим ДЗ (MLP ≈ 88.7% при пороге 88%): свёртки заметно сильнее на изображениях — при похожей постановке задачи CNN легко выходит за 90%.

### Итог по критериям зачёта
- train/test Fashion MNIST скачаны — да;
- архитектура: свёртки + регуляризация + один финальный FC — да (модель `final`);
- качество на тесте **≥ 89,5%** — да (**91.71%**).
